# Import Library

In [159]:
from pynq import Overlay, allocate
import threading
import time
import json, os
import numpy as np

# Import Files

In [160]:
MODEL_PATH = os.path.abspath("model")
DATA_PATH = os.path.abspath("data")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

WEIGHT_MEM_FILE = os.path.join(MODEL_PATH, "G_d5_Q9.14_decoder_weight.mem")
BIAS_MEM_FILE   = os.path.join(MODEL_PATH, "G_d5_Q9.14_decoder_bias.mem")
IFMAP_L0_FILE   = os.path.join(DATA_PATH, "b3.mem")
IFMAP_L1_FILE   = os.path.join(DATA_PATH, "d2_in.mem")
IFMAP_L2_FILE   = os.path.join(DATA_PATH, "d3_in.mem")
IFMAP_L3_FILE   = os.path.join(DATA_PATH, "d4_in.mem")

L0_RESULT   = os.path.join(DATA_PATH, "d1.mem")
L1_RESULT   = os.path.join(DATA_PATH, "d2_out.mem")
L2_RESULT   = os.path.join(DATA_PATH, "d3_out.mem")
L3_RESULT   = os.path.join(DATA_PATH, "d4_out.mem")

# ----- Offset dalam file .mem gabungan -----
WEIGHT_OFFSET_L0 = 0
WEIGHT_OFFSET_L1 = 131072
WEIGHT_OFFSET_L2 = 196608
WEIGHT_OFFSET_L3 = 212992

BIAS_OFFSET_L0 = 0
BIAS_OFFSET_L1 = 8192
BIAS_OFFSET_L2 = 16384
BIAS_OFFSET_L3 = 24576

# Ensuring Correct Data

In [161]:
def load_mem_file(filepath):
    """Load Verilog $readmemh .mem file → numpy uint32 array."""
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.split('//')[0].strip()
            if not line or line.startswith('@'):
                continue
            values.append(int(line, 16))
    return np.array(values, dtype=np.uint32)

In [162]:
print("Loading weight...", end=' ', flush=True)
weight_data = load_mem_file(WEIGHT_MEM_FILE)
print(f"{len(weight_data)} words")

print("Loading bias...", end=' ', flush=True)
bias_data = load_mem_file(BIAS_MEM_FILE)
print(f"{len(bias_data)} words")

print("Loading ifmap L0...", end=' ', flush=True)
ifmap_l0 = load_mem_file(IFMAP_L0_FILE)
print(f"{len(ifmap_l0)} words")

print("Loading ifmap L1...", end=' ', flush=True)
ifmap_l1 = load_mem_file(IFMAP_L1_FILE)
print(f"{len(ifmap_l1)} words")

print("Loading ifmap L2...", end=' ', flush=True)
ifmap_l2 = load_mem_file(IFMAP_L2_FILE)
print(f"{len(ifmap_l2)} words")

print("Loading ifmap L3...", end=' ', flush=True)
ifmap_l3 = load_mem_file(IFMAP_L3_FILE)
print(f"{len(ifmap_l3)} words")

print("Loading L0 Result...", end=' ', flush=True)
result_l0 = load_mem_file(L0_RESULT)
print(f"{len(result_l0)} words")

print("Loading L1 Result...", end=' ', flush=True)
result_l1 = load_mem_file(L1_RESULT)
print(f"{len(result_l1)} words")

print("Loading L2 Result...", end=' ', flush=True)
result_l2 = load_mem_file(L2_RESULT)
print(f"{len(result_l2)} words")

print("Loading L3 Result...", end=' ', flush=True)
result_l3 = load_mem_file(L3_RESULT)
print(f"{len(result_l3)} words")

Loading weight... 217088 words
Loading bias... 32768 words
Loading ifmap L0... 8192 words
Loading ifmap L1... 16384 words
Loading ifmap L2... 16384 words
Loading ifmap L3... 16384 words
Loading L0 Result... 8192 words
Loading L1 Result... 8192 words
Loading L2 Result... 8192 words
Loading L3 Result... 8192 words


In [163]:
def load_mem_file_2d(filepath, channels, time_len, dtype=np.uint32):
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.split('//')[0].strip()
            if not line or line.startswith('@'):
                continue
            values.append(int(line, 16))

    arr = np.array(values, dtype=dtype)

    expected_size = channels * time_len
    if arr.size != expected_size:
        raise ValueError(
            f"Size mismatch in {filepath}: "
            f"expected {expected_size}, got {arr.size}"
        )

    # reshape to (time, channel)
    arr = arr.reshape((time_len, channels))

    # transpose to (channel, time)
    arr = arr.T

    return arr

In [164]:
ch_pos_per_layer = [
    (128, 64),
    (64, 128),
    (32, 256),
    (16, 512)
]

# ---- Load IFMAPS ----

print("Loading IFMAP L0 2D...", flush=True)
ifmap0_2d = load_mem_file_2d(IFMAP_L0_FILE, 256, 32)
print("Shape:", ifmap0_2d.shape)

print("Loading IFMAP L1 2D...", flush=True)
ifmap1_2d = load_mem_file_2d(IFMAP_L1_FILE, 256, 64)  # doubled channel
print("Shape:", ifmap1_2d.shape)

print("Loading IFMAP L2 2D...", flush=True)
ifmap2_2d = load_mem_file_2d(IFMAP_L2_FILE, 128, 128)   # doubled channel
print("Shape:", ifmap2_2d.shape)

print("Loading IFMAP L3 2D...", flush=True)
ifmap3_2d = load_mem_file_2d(IFMAP_L3_FILE, 64, 256)   # doubled channel
print("Shape:", ifmap3_2d.shape)


# ---- Load RESULTS ----

print("Loading RESULT L0 2D...", flush=True)
result0_2d = load_mem_file_2d(L0_RESULT, 128, 64)
print("Shape:", result0_2d.shape)

print("Loading RESULT L1 2D...", flush=True)
result1_2d = load_mem_file_2d(L1_RESULT, 64, 128)
print("Shape:", result1_2d.shape)

print("Loading RESULT L2 2D...", flush=True)
result2_2d = load_mem_file_2d(L2_RESULT, 32, 256)
print("Shape:", result2_2d.shape)

print("Loading RESULT L3 2D...", flush=True)
result3_2d = load_mem_file_2d(L3_RESULT, 16, 512)
print("Shape:", result3_2d.shape)

Loading IFMAP L0 2D...
Shape: (256, 32)
Loading IFMAP L1 2D...
Shape: (256, 64)
Loading IFMAP L2 2D...
Shape: (128, 128)
Loading IFMAP L3 2D...
Shape: (64, 256)
Loading RESULT L0 2D...
Shape: (128, 64)
Loading RESULT L1 2D...
Shape: (64, 128)
Loading RESULT L2 2D...
Shape: (32, 256)
Loading RESULT L3 2D...
Shape: (16, 512)


In [165]:
def compare_prefix_2d(ifmap_2d, result_2d, name=""):
    """
    Compare first half channels of ifmap with result.
    Assumes:
        ifmap shape = (2C, T)
        result shape = (C, T)
    """

    C_res, T_res = result_2d.shape
    C_if, T_if = ifmap_2d.shape

    if T_res != T_if:
        print(f"[{name}] [ERROR] Time dimension mismatch")
        return False

    if C_if < C_res:
        print(f"[{name}] [ERROR] Ifmap has fewer channels than result")
        return False

    prefix = ifmap_2d[:C_res, :]

    if np.array_equal(prefix, result_2d):
        print(f"[{name}] [OPTIMAL] Prefix matches perfectly")
        return True
    else:
        diff = np.where(prefix != result_2d)
        print(f"[{name}] [ERROR] Mismatch at {len(diff[0])} positions")
        print("First 10 mismatches:")
        for i in range(min(10, len(diff[0]))):
            c = diff[0][i]
            t = diff[1][i]
            print(f"  (ch={c}, t={t}) "
                  f"ifmap={prefix[c,t]:08X}, "
                  f"result={result_2d[c,t]:08X}")
        return False

In [166]:
def get_appended_channels_2d(ifmap_2d, result_2d):
    """
    Extract appended channels from ifmap.
    Returns channels beyond result channels.
    """

    C_res = result_2d.shape[0]
    appended = ifmap_2d[C_res:, :]
    return appended

In [167]:
# L1 vs L0
compare_prefix_2d(ifmap1_2d, result0_2d, "L1 vs L0")
append1 = get_appended_channels_2d(ifmap1_2d, result0_2d)
print("L1 appended shape:", append1.shape)


# L2 vs L1
compare_prefix_2d(ifmap2_2d, result1_2d, "L2 vs L1")
append2 = get_appended_channels_2d(ifmap2_2d, result1_2d)
print("L2 appended shape:", append2.shape)


# L3 vs L2
compare_prefix_2d(ifmap3_2d, result2_2d, "L3 vs L2")
append3 = get_appended_channels_2d(ifmap3_2d, result2_2d)
print("L3 appended shape:", append3.shape)

[L1 vs L0] [OPTIMAL] Prefix matches perfectly
L1 appended shape: (128, 64)
[L2 vs L1] [OPTIMAL] Prefix matches perfectly
L2 appended shape: (64, 128)
[L3 vs L2] [OPTIMAL] Prefix matches perfectly
L3 appended shape: (32, 256)


# Loading Hardware

In [169]:
dma_weight = overlay.axi_dma_0
dma_ifmap  = overlay.axi_dma_1
dma_bias   = overlay.axi_dma_2

# Identify the "Listener" channels
dma_receipt = dma_weight.recvchannel 
dma_results = dma_ifmap.recvchannel

# Utility Data Load to FPGA

## Packet Builder

In [170]:
# ============================================================
# CELL 4: Helper — Packet Builder -> OUT NP ARRAY
# ============================================================

def _header(layer_id, num_words_per_bram):
    header = [0x00C0DE, 0x000001, layer_id, 15, 0, int(num_words_per_bram)]
    return [val & 0xFFFFFF for val in header]

# ---- WEIGHT packets ----

def build_weight_l0(batch_id, weight):
    """
    Layer 0 weight packet, 1 batch.
    Per batch: 16 BRAMs × 1024 words = 16384 words.
    OC base = batch_id × 16.
    DDR: OFFSET_L0 + (oc × 1024) + (k × 256) + ich
    """
    buf = _header(0, 1024)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(4):
            oc_abs = batch_id * 16 + tile * 4 + oc_in_tile
            for ich in range(256):
                addr = WEIGHT_OFFSET_L0 + oc_abs * 1024 + k_pos * 256 + ich
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)


# ---- IFMAP packets ----

def build_ifmap_l0(ifmap):
    """
    Layer 0 IFMAP: 32pos × 256ch = 8192 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16] × 256ch.
    DDR layout: pos × 256 + ch
    """
    buf = _header(0, 512)   # 512 words per BRAM
    for bram_id in range(16):
        for pos_group in range(2):
            position = bram_id + pos_group * 16
            for ch in range(256):
                idx = position * 256 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)


# ---- BIAS packets ----

def build_bias_l0(bias):
    """
    Layer 0 bias: 128ch × 64pos = 8192 words.
    BRAM_n: Ch[n, n+16, ..., n+112] (8 page) × 64pos.
    DDR: OFFSET_L0 + ch × 64 + pos
    """
    buf = _header(0, 512)
    for bram_id in range(16):
        for page in range(8):
            ch = page * 16 + bram_id
            for pos in range(64):
                idx = BIAS_OFFSET_L0 + ch * 64 + pos
                buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

print("Packet builder functions defined")

Packet builder functions defined


In [171]:
# ============================================================
# CELL 7: Pre-build semua paket
# ============================================================
import time

t0 = time.time()
print("Building packets...")

# Layer 0 — 8 batches weight
print("  Layer 0 weights (8 batches)...", end=' ', flush=True)
pkt_w_l0 = [build_weight_l0(b, weight_data) for b in range(8)]
print(f"{sum(len(p) for p in pkt_w_l0):,} words total")

print("  Layer 0 ifmap...", end=' ', flush=True)
pkt_i_l0 = build_ifmap_l0(ifmap_l0)
print(f"{len(pkt_i_l0):,} words")

print("  Layer 0 bias...", end=' ', flush=True)
pkt_b_l0 = build_bias_l0(bias_data)
print(f"{len(pkt_b_l0):,} words")

print(f"\nBuild selesai dalam {time.time()-t0:.1f} detik")

Building packets...
  Layer 0 weights (8 batches)... 131,120 words total
  Layer 0 ifmap... 8,198 words
  Layer 0 bias... 8,198 words

Build selesai dalam 1.3 detik


## Sending Utility

In [172]:
def dma_send_async(dma, packet_np):
    '''
    == Example Usage == 
    buf_handle = dma_send_async(dma_weight, pkt_w_l0[0])
    dma_weight.sendchannel.wait() 
    buf_handle.freebuffer()
    '''
    buf = allocate(shape=(len(packet_np),), dtype=np.uint32)
    np.copyto(buf, packet_np)

    dma.sendchannel.transfer(buf)
    
    return buf

def to_signed_24bit(arr):
    """Convert 24-bit unsigned integers to signed 32-bit integers."""
    # Mask to ensure only 24 bits are considered
    a = np.array(arr, dtype=np.int32) & 0xFFFFFF
    # If the 24th bit is 1, subtract 2^24 to get the negative value
    return np.where(a >= 0x800000, a - 0x1000000, a)

# Execute

In [173]:
def reset_dma(dma):
    # 1. Hardware Reset (bit 2 of Control Register)
    # Offset 0x00 for Send (MM2S), Offset 0x30 for Receive (S2MM)
    dma.write(0x00, 0x4)  
    dma.write(0x30, 0x4)
    
    # 2. Wait for reset bit to clear (hardware clears it when done)
    timeout = 0
    while (dma.read(0x00) & 0x4) and timeout < 1000:
        timeout += 1
    
    # 3. CRITICAL: Restart the DMA channels
    # PYNQ channels have a .start() method that sets the RS (Run/Stop) bit
    dma.sendchannel.start()
    dma.recvchannel.start()
    
    print(f"DMA {dma} reset and restarted.")

In [174]:
for d in [dma_weight, dma_ifmap]:
    reset_dma(d)

# 1. Prepare buffers for the Receipts (6 words) and the Final Results (4102 words)
receipt_buf = allocate(shape=(6,), dtype=np.uint32)
results_buf_m0 = allocate(shape=(8204,), dtype=np.uint32)
results_buf_m1 = allocate(shape=(6,), dtype=np.uint32)

print("===== STARTING LAYER 0 EXECUTION =====")

# 2. SEND STATIC DATA (Bias and IFMAP)
print("Sending Bias and IFMAP...")
h_bias  = dma_send_async(dma_bias, pkt_b_l0)
h_ifmap = dma_send_async(dma_ifmap, pkt_i_l0)
dma_bias.sendchannel.wait()
dma_ifmap.sendchannel.wait()
h_bias.freebuffer()
h_ifmap.freebuffer()

# 3. WEIGHT BATCH LOOP
for b in range(8):
    print(f"--- Processing Batch {b} ---")
    receipt_buf[0] = 0x0
    
    # Listen
    dma_receipt.transfer(receipt_buf)
    h_w = dma_send_async(dma_weight, pkt_w_l0[b])

    # Clean up the send side
    # If the DMA is halted, .wait() will crash. Let's check 'running' first.
#     if dma_weight.sendchannel.running:
    dma_weight.sendchannel.wait()
    h_w.freebuffer()

# 4. COLLECT FINAL RESULTS
print("All batches sent. Waiting for Final DA7A Output...")

# Arm both channels for the final data dump
dma_receipt.transfer(results_buf_m0)
dma_results.transfer(results_buf_m1)

# Diagnostic loop
print("Monitoring Final Transfer...")
OFFSET_SR = 0x34
OFFSET_LEN = 0x58

for _ in range(5):
    len0 = dma_weight.read(OFFSET_LEN)
    len1 = dma_ifmap.read(OFFSET_LEN)
    idle0 = (dma_weight.read(OFFSET_SR) & 0x2) >> 1
    idle1 = (dma_ifmap.read(OFFSET_SR) & 0x2) >> 1
    
    print(f"  M0: {len0} bytes (Idle:{idle0}) | M1: {len1} bytes (Idle:{idle1})")
    if idle0 and idle1: break
    time.sleep(1)

# Instead of dma_receipt.wait() which hangs forever:
print("Force-stopping DMA to rescue data...")
dma_receipt.stop()
dma_results.stop()



# 1. Check the Magic Numbers
magic_start = hex(results_buf_m0[0] & 0xFFFF)
# The second header should be at index 4102 (6 header + 4096 data)
magic_middle = hex(results_buf_m0[4102] & 0xFFFF)

print(f"Header 1 Magic: {magic_start}")
print(f"Header 2 Magic: {magic_middle}")

# 2. Verify Data Content
# Let's see if there are actual non-zero values in the data sections
data_section_1 = results_buf_m0[6:4102]
data_section_2 = results_buf_m0[4108:8204]

print(f"Section 1 Non-zeros: {np.count_nonzero(data_section_1)}")
print(f"Section 2 Non-zeros: {np.count_nonzero(data_section_2)}")

if magic_start == '0xda7a' and magic_middle == '0xda7a':
    print("\nSUCCESS: Both data packets recovered from M0 stream!")


DMA <pynq.lib.dma.DMA object at 0xaa11d4d8> reset and restarted.
DMA <pynq.lib.dma.DMA object at 0xad2e9238> reset and restarted.
===== STARTING LAYER 0 EXECUTION =====
Sending Bias and IFMAP...
--- Processing Batch 0 ---
--- Processing Batch 1 ---
--- Processing Batch 2 ---
--- Processing Batch 3 ---
--- Processing Batch 4 ---
--- Processing Batch 5 ---
--- Processing Batch 6 ---
--- Processing Batch 7 ---
All batches sent. Waiting for Final DA7A Output...
Monitoring Final Transfer...
  M0: 32816 bytes (Idle:0) | M1: 24 bytes (Idle:1)
  M0: 32816 bytes (Idle:0) | M1: 24 bytes (Idle:1)
  M0: 32816 bytes (Idle:0) | M1: 24 bytes (Idle:1)
  M0: 32816 bytes (Idle:0) | M1: 24 bytes (Idle:1)
  M0: 32816 bytes (Idle:0) | M1: 24 bytes (Idle:1)
Force-stopping DMA to rescue data...


KeyboardInterrupt: 

In [ ]:
# def check_dma_error(dma, name):
#     # Read S2MM_SR (Status Register) at offset 0x34
#     sr = dma.read(0x34)
#     errors = []
#     if sr & (1<<4): errors.append("DMAInternalError (TLAST early/late?)")
#     if sr & (1<<5): errors.append("DMASlaveError (Axi Bus Blocked)")
#     if sr & (1<<6): errors.append("DMADecodeError (Invalid Address)")
#     if sr & (1<<8): errors.append("SGIntErr")
    
#     print(f"--- {name} STATUS ---")
#     print(f"Raw SR: {hex(sr)}")
#     print(f"Halted: {bool(sr & 0x1)}")
#     print(f"Idle: {bool(sr & 0x2)}")
#     if errors:
#         print(f"ERRORS DETECTED: {errors}")
#     else:
#         print("No errors detected in SR.")

# # Let's see what happened to Batch 1
# check_dma_error(dma_weight, "DMA_WEIGHT (M0/Receipts)")

In [175]:
print("Inspecting the 'frozen' buffer...")

# Even if the DMA crashed, the data is usually already sitting in the buffer
# because the S2MM (Stream to Memory Map) writes as data arrives.

# 1. Check Magic Numbers
m0_start = hex(results_buf_m0[0] & 0xFFFF)
# Index 4102 is the start of the second packet (6 header + 4096 data)
m0_mid   = hex(results_buf_m0[4102] & 0xFFFF)

print(f"Index 0: {m0_start}")
print(f"Index 4102: {m0_mid}")

# 2. Count non-zeros to see if it's junk or real math
nz1 = np.count_nonzero(results_buf_m0[6:4102])
nz2 = np.count_nonzero(results_buf_m0[4108:8204])

print(f"Data Section 1 Non-Zeros: {nz1}")
print(f"Data Section 2 Non-Zeros: {nz2}")

if m0_start == '0xda7a' and m0_mid == '0xda7a':
    print("\nSUCCESS: The data is valid! You don't need the DMA to finish.")

Inspecting the 'frozen' buffer...
Index 0: 0xda7a
Index 4102: 0x0
Data Section 1 Non-Zeros: 77
Data Section 2 Non-Zeros: 0


In [186]:
valid_results = results_buf_m0[6:4102]
print("First 10 non-zero values found:")
print(valid_results[valid_results > 0][:10])
print("First 10 values found:")
print(valid_results[:100])

First 10 non-zero values found:
[15903 10463  4393  3677 41107   665 57279 40232 29023  6576]
First 10 values found:
[    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0 15903     0     0     0     0
     0     0     0     0     0 10463     0     0     0     0     0     0
     0     0     0     0     0     0     0  4393     0  3677     0     0
     0 41107   665 57279     0 40232     0 29023     0  6576     0 10729
 39093 37092     0 15775]


In [187]:
print("First 10 reference values:")
print([x for x in result_l0[:100]])

First 10 reference values:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 393, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1463, 0, 0, 0, 0, 0, 22180, 284, 0, 0, 0, 0, 0, 17561, 0, 0, 0, 7459, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7915, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4742, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19993, 0, 0, 30884, 0, 0, 0, 0, 0, 0, 0]
